In [1]:
!pip install -q -U \
    bitsandbytes==0.50.0 \
    transformers==5.15.0 \
    peft==0.20.0 \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 47.4 MB/s eta 0:00:00


In [2]:
import torch
import bitsandbytes as bnb

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

print("bitsandbytes:", bnb.__version__)

ENVIRONMENT CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8
bitsandbytes: 0.50.0


In [3]:
import transformers
import peft

print("=" * 60)
print("LIBRARY CHECK")
print("=" * 60)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

LIBRARY CHECK
Transformers: 5.15.0
PEFT: 0.20.0


In [4]:
from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted successfully.")

Mounted at /content/drive
✅ Google Drive mounted successfully.


In [5]:
import os

BASE_MODEL = "meta-llama/Llama-3.2-3B-Instruct"

ADAPTER_PATH = (
    "/content/drive/MyDrive/"
    "Sparkling_AI_Model/"
    "sparkling_qlora/"
    "final_adapter"
)

print("=" * 60)
print("MODEL PATH CHECK")
print("=" * 60)

print("Base model:", BASE_MODEL)
print("Adapter path:", ADAPTER_PATH)
print("Adapter exists:", os.path.exists(ADAPTER_PATH))

MODEL PATH CHECK
Base model: meta-llama/Llama-3.2-3B-Instruct
Adapter path: /content/drive/MyDrive/Sparkling_AI_Model/sparkling_qlora/final_adapter
Adapter exists: True


In [6]:
required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json"
]

print("=" * 60)
print("SPARKLING ADAPTER CHECK")
print("=" * 60)

for filename in required_files:
    path = os.path.join(ADAPTER_PATH, filename)

    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {filename} ({size:.2f} MB)")
    else:
        print(f"❌ {filename} NOT FOUND")

SPARKLING ADAPTER CHECK
✅ adapter_model.safetensors (35.03 MB)
✅ adapter_config.json (0.00 MB)
✅ tokenizer.json (16.41 MB)


In [7]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel

import torch

print("=" * 60)
print("MODEL LIBRARIES")
print("=" * 60)
print("✅ Transformers imported")
print("✅ PEFT imported")
print("PyTorch:", torch.__version__)

MODEL LIBRARIES
✅ Transformers imported
✅ PEFT imported
PyTorch: 2.11.0+cu128


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("=" * 60)
print("4-BIT QUANTIZATION")
print("=" * 60)
print("Load in 4-bit: True")
print("Quantization type: NF4")
print("Compute dtype: float16")
print("Double quantization: True")

4-BIT QUANTIZATION
Load in 4-bit: True
Quantization type: NF4
Compute dtype: float16
Double quantization: True


In [ ]:
from huggingface_hub import login
login()

In [10]:
print("=" * 60)
print("LOADING TOKENIZER")
print("=" * 60)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)

print("✅ Tokenizer loaded successfully.")
print("Tokenizer type:", type(tokenizer).__name__)
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

LOADING TOKENIZER


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✅ Tokenizer loaded successfully.
Tokenizer type: TokenizersBackend
EOS token: <|eot_id|>
PAD token: None


In [11]:
print("=" * 60)
print("LOADING LLAMA 3.2 3B")
print("=" * 60)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Base Llama model loaded successfully.")
print("Model type:", type(model).__name__)
print("Model device:", model.device)

LOADING LLAMA 3.2 3B


model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

✅ Base Llama model loaded successfully.
Model type: LlamaForCausalLM
Model device: cuda:0


In [12]:
print("=" * 60)
print("LOADING SPARKLING LORA ADAPTER")
print("=" * 60)

model = PeftModel.from_pretrained(
    model,
    ADAPTER_PATH
)

print("✅ Sparkling LoRA adapter loaded successfully.")
print("Model type:", type(model).__name__)

LOADING SPARKLING LORA ADAPTER
✅ Sparkling LoRA adapter loaded successfully.
Model type: PeftModelForCausalLM


In [13]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

print("=" * 60)
print("TOKENIZER CONFIGURATION")
print("=" * 60)
print("PAD token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("PAD token ID:", tokenizer.pad_token_id)
print("EOS token ID:", tokenizer.eos_token_id)

TOKENIZER CONFIGURATION
PAD token: <|eot_id|>
EOS token: <|eot_id|>
PAD token ID: 128009
EOS token ID: 128009


In [14]:
test_input = {
    "instruction": "Generate context-aware UX microcopy",
    "input": {
        "component_type": "button",
        "component_role": "primary_action",
        "current_text": "Upload",
        "screen_type": "file_upload",
        "ui_context": "Primary task: upload document",
        "nearby_text": "Add a PDF or DOCX document, up to 10 MB.",
        "intent": "upload_document",
        "persona": "experienced_user",
        "tone": "professional",
        "accessibility_requirement": "specific action"
    }
}

print("Test input created:")
print(test_input)

Test input created:
{'instruction': 'Generate context-aware UX microcopy', 'input': {'component_type': 'button', 'component_role': 'primary_action', 'current_text': 'Upload', 'screen_type': 'file_upload', 'ui_context': 'Primary task: upload document', 'nearby_text': 'Add a PDF or DOCX document, up to 10 MB.', 'intent': 'upload_document', 'persona': 'experienced_user', 'tone': 'professional', 'accessibility_requirement': 'specific action'}}


In [15]:
def create_sparkling_prompt(data):
    inp = data["input"]

    prompt = f"""### Instruction:

Generate context-aware UX microcopy.

Generate ONE concise UX microcopy suggestion.
Return ONLY the suggested microcopy.
Do NOT provide a rationale.
Do NOT explain your answer.
Do NOT provide multiple suggestions.

### Input:

Component type: {inp["component_type"]}
Component role: {inp["component_role"]}
Current text: {inp["current_text"]}
Screen type: {inp["screen_type"]}
UI context: {inp["ui_context"]}
Nearby text: {inp["nearby_text"]}
Intent: {inp["intent"]}
Persona: {inp["persona"]}
Tone: {inp["tone"]}
Accessibility requirement: {inp["accessibility_requirement"]}

### Response:
"""

    return prompt


prompt = create_sparkling_prompt(test_input)

print(prompt)

### Instruction:

Generate context-aware UX microcopy.

Generate ONE concise UX microcopy suggestion.
Return ONLY the suggested microcopy.
Do NOT provide a rationale.
Do NOT explain your answer.
Do NOT provide multiple suggestions.

### Input:

Component type: button
Component role: primary_action
Current text: Upload
Screen type: file_upload
UI context: Primary task: upload document
Nearby text: Add a PDF or DOCX document, up to 10 MB.
Intent: upload_document
Persona: experienced_user
Tone: professional
Accessibility requirement: specific action

### Response:



In [16]:
import torch

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Move inputs to the same GPU as the model
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

print("Generating Sparkling microcopy...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        temperature=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

# Decode only the newly generated tokens
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

prediction = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("=" * 60)
print("SPARKLING PREDICTION")
print("=" * 60)
print(prediction)

Generating Sparkling microcopy...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


SPARKLING PREDICTION
Add document


In [17]:
test_cases = [
    {
        "id": "UX-0096",
        "input": {
            "component_type": "button",
            "component_role": "primary_action",
            "current_text": "Upload",
            "screen_type": "file_upload",
            "ui_context": "Primary task: upload document",
            "nearby_text": "Add a PDF or DOCX document, up to 10 MB.",
            "intent": "upload_document",
            "persona": "experienced_user",
            "tone": "professional",
            "accessibility_requirement": "specific action"
        },
        "references": [
            "Upload document",
            "Choose a document",
            "Add document"
        ]
    },

    {
        "id": "UX-0207",
        "input": {
            "component_type": "button",
            "component_role": "secondary_action",
            "current_text": "Log Out",
            "screen_type": "account",
            "ui_context": "End current user session",
            "nearby_text": "",
            "intent": "end_session",
            "persona": "general_user",
            "tone": "professional",
            "accessibility_requirement": "clear action"
        },
        "references": [
            "Log out",
            "Sign out",
            "End session"
        ]
    },

    {
        "id": "UX-0393",
        "input": {
            "component_type": "empty_state",
            "component_role": "saved_items",
            "current_text": "No saved items",
            "screen_type": "resources",
            "ui_context": "User has not saved any resources",
            "nearby_text": "",
            "intent": "no_saved_items",
            "persona": "general_user",
            "tone": "friendly",
            "accessibility_requirement": "clear guidance"
        },
        "references": [
            "No saved resources yet",
            "Save a resource to find it here",
            "Your saved resources will appear here"
        ]
    },

    {
        "id": "UX-0398",
        "input": {
            "component_type": "empty_state",
            "component_role": "saved_items",
            "current_text": "No saved items",
            "screen_type": "resources",
            "ui_context": "User has not saved any resources",
            "nearby_text": "",
            "intent": "no_saved_items",
            "persona": "general_user",
            "tone": "encouraging",
            "accessibility_requirement": "clear guidance"
        },
        "references": [
            "No saved resources yet",
            "Save a resource to find it here",
            "Your saved resources will appear here"
        ]
    },

    {
        "id": "UX-0099",
        "input": {
            "component_type": "button",
            "component_role": "primary_action",
            "current_text": "Submit",
            "screen_type": "feedback",
            "ui_context": "Submit user feedback",
            "nearby_text": "",
            "intent": "submit_feedback",
            "persona": "general_user",
            "tone": "warm",
            "accessibility_requirement": "clear action"
        },
        "references": [
            "Send feedback",
            "Submit feedback",
            "Share feedback"
        ]
    }
]

print("Test cases:", len(test_cases))

Test cases: 5


In [18]:
def generate_microcopy(data):
    prompt = create_sparkling_prompt({
        "input": data["input"]
    })

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return prediction


print("=" * 70)
print("SPARKLING MODEL — 5 CASE RELOAD TEST")
print("=" * 70)

for case in test_cases:
    prediction = generate_microcopy(case)

    print(f"\nID: {case['id']}")
    print(f"Current text: {case['input']['current_text']}")
    print("\nReference answers:")

    for ref in case["references"]:
        print(f"- {ref}")

    print(f"\nRELOADED MODEL OUTPUT:")
    print(prediction)

SPARKLING MODEL — 5 CASE RELOAD TEST

ID: UX-0096
Current text: Upload

Reference answers:
- Upload document
- Choose a document
- Add document

RELOADED MODEL OUTPUT:
Add document

ID: UX-0207
Current text: Log Out

Reference answers:
- Log out
- Sign out
- End session

RELOADED MODEL OUTPUT:
End session

ID: UX-0393
Current text: No saved items

Reference answers:
- No saved resources yet
- Save a resource to find it here
- Your saved resources will appear here

RELOADED MODEL OUTPUT:
No saved resources yet

ID: UX-0398
Current text: No saved items

Reference answers:
- No saved resources yet
- Save a resource to find it here
- Your saved resources will appear here

RELOADED MODEL OUTPUT:
No saved resources yet

ID: UX-0099
Current text: Submit

Reference answers:
- Send feedback
- Submit feedback
- Share feedback

RELOADED MODEL OUTPUT:
Submit feedback


In [19]:
import os

TEST_PATH = (
    "/content/drive/MyDrive/"
    "Sparkling_AI_Model/"
    "dataset/"
    "test_v2.jsonl"
)

print("Test file exists:", os.path.exists(TEST_PATH))
print("Test path:", TEST_PATH)

Test file exists: False
Test path: /content/drive/MyDrive/Sparkling_AI_Model/dataset/test_v2.jsonl


In [20]:
import os

print("=" * 60)
print("SEARCHING GOOGLE DRIVE FOR test_v2.jsonl")
print("=" * 60)

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.lower() == "test_v2.jsonl":
            matches.append(os.path.join(root, file))

if matches:
    print(f"\n✅ Found {len(matches)} file(s):\n")
    for path in matches:
        print(path)
else:
    print("\n❌ test_v2.jsonl was not found in My Drive.")

SEARCHING GOOGLE DRIVE FOR test_v2.jsonl

❌ test_v2.jsonl was not found in My Drive.


In [21]:
TEST_PATH = (
    "/content/drive/MyDrive/"
    "Sparkling_AI_Model/"
    "dataset/"
    "test.jsonl"
)

import os

print("Test file exists:", os.path.exists(TEST_PATH))

if os.path.exists(TEST_PATH):
    print("✅ test.jsonl found")

Test file exists: True
✅ test.jsonl found


In [22]:
import json
import os

print("=" * 60)
print("LOADING TEST DATASET")
print("=" * 60)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_records = [json.loads(line) for line in f if line.strip()]

print("Number of test records:", len(test_records))

print("\nFirst record:")
print(json.dumps(test_records[0], indent=2, ensure_ascii=False))

LOADING TEST DATASET
Number of test records: 144

First record:
{
  "text": "### Instruction:\nGenerate context-aware UX microcopy.\n\nGenerate ONE concise UX microcopy suggestion.\nReturn ONLY the suggested microcopy.\nDo NOT provide a rationale.\nDo NOT explain your answer.\nDo NOT provide multiple suggestions.\n\n### Input:\nComponent type: button\nComponent role: primary_action\nCurrent text: Upload\nScreen type: file_upload\nUI context: Primary task: upload document\nNearby text: Add a PDF or DOCX document, up to 10 MB.\nIntent: upload_document\nPersona: experienced_user\nTone: professional\nAccessibility requirement: specific action\n\n### Response:\nUpload document"
}


In [23]:
print("=" * 60)
print("PREPARING TEST DATA FOR EVALUATION")
print("=" * 60)

evaluation_records = []

for record in test_records:
    text = record["text"]

    # Split training example into prompt and reference answer
    if "### Response:" not in text:
        print("⚠️ Missing ### Response: in one record")
        continue

    prompt_text, reference_text = text.split(
        "### Response:",
        1
    )

    prompt_text = prompt_text.strip()
    reference_text = reference_text.strip()

    evaluation_records.append({
        "prompt": prompt_text + "\n\n### Response:\n",
        "reference": reference_text
    })

print("Total evaluation records:", len(evaluation_records))

print("\nFirst evaluation prompt:")
print(evaluation_records[0]["prompt"])

print("\nFirst reference answer:")
print(evaluation_records[0]["reference"])

PREPARING TEST DATA FOR EVALUATION
Total evaluation records: 144

First evaluation prompt:
### Instruction:
Generate context-aware UX microcopy.

Generate ONE concise UX microcopy suggestion.
Return ONLY the suggested microcopy.
Do NOT provide a rationale.
Do NOT explain your answer.
Do NOT provide multiple suggestions.

### Input:
Component type: button
Component role: primary_action
Current text: Upload
Screen type: file_upload
UI context: Primary task: upload document
Nearby text: Add a PDF or DOCX document, up to 10 MB.
Intent: upload_document
Persona: experienced_user
Tone: professional
Accessibility requirement: specific action

### Response:


First reference answer:
Upload document


In [24]:
import json
import os
import torch
import time

RESULTS_PATH = (
    "/content/drive/MyDrive/"
    "Sparkling_AI_Model/"
    "sparkling_qlora/"
    "phase7_predictions.json"
)

predictions = []

print("=" * 70)
print("SPARKLING PHASE 7 — FULL TEST SET INFERENCE")
print("=" * 70)
print("Total records:", len(evaluation_records))
print("Results will be saved to:")
print(RESULTS_PATH)
print()

start_time = time.time()

for i, record in enumerate(evaluation_records):

    inputs = tokenizer(
        record["prompt"],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    predictions.append({
        "index": i,
        "prediction": prediction,
        "reference": record["reference"]
    })

    # Save progress every 10 records
    if (i + 1) % 10 == 0 or (i + 1) == len(evaluation_records):

        with open(
            RESULTS_PATH,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                predictions,
                f,
                indent=2,
                ensure_ascii=False
            )

        elapsed = time.time() - start_time

        print(
            f"Progress: {i + 1}/{len(evaluation_records)} "
            f"| Elapsed: {elapsed / 60:.1f} min"
        )

print()
print("=" * 70)
print("✅ INFERENCE COMPLETED")
print("=" * 70)
print("Predictions:", len(predictions))
print("Saved to:", RESULTS_PATH)

SPARKLING PHASE 7 — FULL TEST SET INFERENCE
Total records: 144
Results will be saved to:
/content/drive/MyDrive/Sparkling_AI_Model/sparkling_qlora/phase7_predictions.json

Progress: 10/144 | Elapsed: 0.1 min
Progress: 20/144 | Elapsed: 0.1 min
Progress: 30/144 | Elapsed: 0.2 min
Progress: 40/144 | Elapsed: 0.3 min
Progress: 50/144 | Elapsed: 0.4 min
Progress: 60/144 | Elapsed: 0.4 min
Progress: 70/144 | Elapsed: 0.5 min
Progress: 80/144 | Elapsed: 0.5 min
Progress: 90/144 | Elapsed: 0.6 min
Progress: 100/144 | Elapsed: 0.7 min
Progress: 110/144 | Elapsed: 0.7 min
Progress: 120/144 | Elapsed: 0.8 min
Progress: 130/144 | Elapsed: 0.9 min
Progress: 140/144 | Elapsed: 1.0 min
Progress: 144/144 | Elapsed: 1.0 min

✅ INFERENCE COMPLETED
Predictions: 144
Saved to: /content/drive/MyDrive/Sparkling_AI_Model/sparkling_qlora/phase7_predictions.json


In [25]:
import json

with open(
    RESULTS_PATH,
    "r",
    encoding="utf-8"
) as f:
    predictions = json.load(f)

def normalize_text(text):
    return " ".join(text.lower().strip().split())

exact_matches = 0

for item in predictions:
    prediction = normalize_text(item["prediction"])
    reference = normalize_text(item["reference"])

    if prediction == reference:
        exact_matches += 1

exact_match_score = (
    exact_matches / len(predictions)
) * 100

print("=" * 60)
print("FINE-TUNED MODEL — EXACT MATCH")
print("=" * 60)

print(f"Exact matches: {exact_matches}/{len(predictions)}")
print(f"Exact Match: {exact_match_score:.2f}%")

FINE-TUNED MODEL — EXACT MATCH
Exact matches: 48/144
Exact Match: 33.33%


In [27]:
!pip install -q rouge-score

  Preparing metadata (setup.py) ... done


In [28]:
from rouge_score import rouge_scorer

print("=" * 60)
print("FINE-TUNED MODEL — ROUGE-L")
print("=" * 60)

scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

rouge_scores = []

for item in predictions:
    prediction = item["prediction"]
    reference = item["reference"]

    score = scorer.score(
        reference,
        prediction
    )["rougeL"].fmeasure

    rouge_scores.append(score)

rouge_l_score = (
    sum(rouge_scores) / len(rouge_scores)
) * 100

print(f"ROUGE-L: {rouge_l_score:.2f}%")
print(f"Records evaluated: {len(rouge_scores)}")

FINE-TUNED MODEL — ROUGE-L
ROUGE-L: 58.83%
Records evaluated: 144


In [29]:
print("Semantic similarity function:", calculate_semantic_similarity)

NameError: name 'calculate_semantic_similarity' is not defined

In [30]:
!pip install -q sentence-transformers scikit-learn

In [31]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print("=" * 60)
print("LOADING SEMANTIC SIMILARITY MODEL")
print("=" * 60)

semantic_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("✅ Semantic model loaded")

LOADING SEMANTIC SIMILARITY MODEL


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Semantic model loaded


In [32]:
print("=" * 60)
print("FINE-TUNED MODEL — SEMANTIC SIMILARITY")
print("=" * 60)

predicted_texts = [
    item["prediction"]
    for item in predictions
]

reference_texts = [
    item["reference"]
    for item in predictions
]

prediction_embeddings = semantic_model.encode(
    predicted_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

reference_embeddings = semantic_model.encode(
    reference_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

similarities = []

for pred_embedding, ref_embedding in zip(
    prediction_embeddings,
    reference_embeddings
):
    similarity = cosine_similarity(
        [pred_embedding],
        [ref_embedding]
    )[0][0]

    similarities.append(similarity)

semantic_similarity_score = (
    np.mean(similarities) * 100
)

print()
print("=" * 60)
print(f"Semantic Similarity: {semantic_similarity_score:.2f}%")
print(f"Records evaluated: {len(similarities)}")
print("=" * 60)

FINE-TUNED MODEL — SEMANTIC SIMILARITY


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


Semantic Similarity: 73.21%
Records evaluated: 144


In [33]:
print("=" * 60)
print("FINE-TUNED MODEL — SEMANTIC SIMILARITY RESULT")
print("=" * 60)

print(f"Semantic Similarity: {semantic_similarity_score:.2f}%")
print(f"Records evaluated: {len(similarities)}")
print("=" * 60)

FINE-TUNED MODEL — SEMANTIC SIMILARITY RESULT
Semantic Similarity: 73.21%
Records evaluated: 144


In [34]:
import json
import os

final_results = {
    "evaluation_dataset": "test.jsonl",
    "test_records": 144,

    "baseline": {
        "exact_match": 6.25,
        "rouge_l": 53.64,
        "semantic_similarity": 68.13
    },

    "fine_tuned": {
        "exact_match": 33.33,
        "rouge_l": 58.83,
        "semantic_similarity": 73.21
    },

    "improvement_percentage_points": {
        "exact_match": 33.33 - 6.25,
        "rouge_l": 58.83 - 53.64,
        "semantic_similarity": 73.21 - 68.13
    }
}

FINAL_RESULTS_PATH = (
    "/content/drive/MyDrive/"
    "Sparkling_AI_Model/"
    "sparkling_qlora/"
    "phase7_final_results.json"
)

with open(
    FINAL_RESULTS_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_results,
        f,
        indent=2
    )

print("=" * 70)
print("SPARKLING PHASE 7 — FINAL RESULTS")
print("=" * 70)

print(f"Exact Match:          6.25% → 33.33% (+27.08 pp)")
print(f"ROUGE-L:              53.64% → 58.83% (+5.19 pp)")
print(f"Semantic Similarity:  68.13% → 73.21% (+5.08 pp)")

print()
print("Test records:", 144)
print("✅ Final results saved")
print(FINAL_RESULTS_PATH)

SPARKLING PHASE 7 — FINAL RESULTS
Exact Match:          6.25% → 33.33% (+27.08 pp)
ROUGE-L:              53.64% → 58.83% (+5.19 pp)
Semantic Similarity:  68.13% → 73.21% (+5.08 pp)

Test records: 144
✅ Final results saved
/content/drive/MyDrive/Sparkling_AI_Model/sparkling_qlora/phase7_final_results.json
